# UGC Ad Factory — Google Colab (main pipeline)

Runs the **full pipeline in the cloud**: script (Ollama + Qwen 2.5 7B, same as local), voice (gTTS), captions (faster-whisper), Pexels images, Ken Burns clips, and FFmpeg assembly. No external LLM API — uses Ollama in Colab to avoid rate limits and API keys. Heavy steps (image processing, clip generation, animations, FFmpeg) run in Colab for better performance.

**Before running:**
1. Run the **Install Ollama** cell first (installs Ollama + pulls the model)
2. Get a **Pexels API key** (optional): https://www.pexels.com/api/ — for real B-roll images; otherwise placeholders
3. Run cells in order. Optionally enable GPU for faster Whisper.

## 1. Install dependencies (pip + FFmpeg)

## 2. Install Ollama + pull model

Run this once per session. Uses the same model as the local pipeline (qwen2.5:7b-instruct-q4_K_M). No API key needed. The pip/apt cell above must run first.

In [ ]:
# Pip packages (run once per session)
!pip install -q requests gtts faster-whisper Pillow

# FFmpeg for video rendering
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null

print("Dependencies installed.")

In [ ]:
# Install Ollama (same as local)
!curl -fsSL https://ollama.com/install.sh | sh

# Start server in background
import subprocess, time
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
time.sleep(3)

# Pull model (same as local: qwen2.5:7b-instruct-q4_K_M)
!ollama pull qwen2.5:7b-instruct-q4_K_M

print("Ollama ready.")

## 2. Add pipeline code

Upload **`colab_pipeline.py`** from this repo into Colab (drag into the Files panel on the left), or clone the repo and use the file from `prototype-macos/colab/colab_pipeline.py`.

In [ ]:
# Option A: Clone the repo (replace with your repo URL)
# !git clone https://github.com/YOUR_USER/ugc-ad-factory.git
# import sys; sys.path.insert(0, "/content/ugc-ad-factory/prototype-macos/colab")

# Option B: If you uploaded colab_pipeline.py into /content, it's already on the path
import sys
for p in ["/content", "/content/ugc-ad-factory/prototype-macos/colab"]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    import colab_pipeline
    print("colab_pipeline loaded.")
except ImportError:
    print("Upload colab_pipeline.py (left sidebar → Files → Upload) or clone the repo, then re-run.")

## 3. Set API keys and product inputs

In [ ]:
import os

# Optional: Pexels API key for real B-roll images (https://www.pexels.com/api/)
# Leave empty to use placeholders only
os.environ["PEXELS_API_KEY"] = "YOUR_PEXELS_API_KEY"

# Product brief
PRODUCT = "AI astrology app"
DESCRIPTION = "Daily predictions using your birth chart"
AUDIENCE = "Young professionals interested in astrology"

# Optional: Whisper model (tiny, base, small, medium) — base saves RAM
WHISPER_MODEL = "base"
# Optional: Ollama model (same as local default)
OLLAMA_MODEL = "qwen2.5:7b-instruct-q4_K_M"

## 4. Run the full pipeline

In [ ]:
from colab_pipeline import run_full_pipeline

paths = run_full_pipeline(
    product_name=PRODUCT,
    product_description=DESCRIPTION,
    target_audience=AUDIENCE,
    whisper_model=WHISPER_MODEL,
    ollama_model=OLLAMA_MODEL,
)

print("Outputs:", paths)

## 5. Download the final video

In [ ]:
from google.colab import files

final_mp4 = paths["final_mp4"]
if final_mp4.exists():
    files.download(str(final_mp4))
else:
    print("final.mp4 not found.")